In [29]:
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.4f}".format

from Scripts.weekly_view import build_weekly_view
from Scripts.schedule import build_season_schedule
from Scripts.data_io import load_odds_and_results

PROJECT_ROOT = Path("/")

YEARS = [2024, 2025]

# Output master dataset (new file)
OUT_PATH = PROJECT_ROOT / "Data" / "in Use" / "oad_analysis_master_2024_2025.csv"

In [26]:
odds_df = load_odds_and_results()

# Normalize column name for tier
tier_col = None
for c in ["Event_Tier", "event_tier", "tier", "Tier"]:
    if c in odds_df.columns:
        tier_col = c
        break

if tier_col is None:
    raise ValueError("Odds_and_Results has no Event_Tier/event_tier column.")

tier_map = odds_df.copy()
tier_map["year"] = pd.to_numeric(tier_map["year"], errors="coerce").astype("Int64")
tier_map["event_id"] = pd.to_numeric(tier_map["event_id"], errors="coerce").astype("Int64")

tier_map = (
    tier_map[["year", "event_id", tier_col]]
    .dropna(subset=["year","event_id"])
    .drop_duplicates(subset=["year","event_id"])
    .rename(columns={tier_col: "Event_Tier"})
)

tier_map["Event_Tier"] = tier_map["Event_Tier"].astype(str).str.strip().str.lower().replace({
    "sig": "signature",
    "signature event": "signature",
    "regular event": "regular",
    "major championship": "major",
    "maj": "major",
})

print(tier_map["Event_Tier"].value_counts(dropna=False))

Event_Tier
regular      50
signature    33
major        12
Name: count, dtype: int64


In [27]:
events = []
for yr in YEARS:
    s = build_season_schedule(yr).copy()
    s["year"] = yr
    # keep only what we need
    keep = ["year", "event_id", "event_name", "event_date"]
    keep = [c for c in keep if c in s.columns]
    s = s[keep].drop_duplicates(subset=["year","event_id"])
    events.append(s)

events_df = pd.concat(events, ignore_index=True)
events_df["event_id"] = pd.to_numeric(events_df["event_id"], errors="coerce").astype("Int64")
events_df = events_df.dropna(subset=["event_id"])

print("Events:", len(events_df))
display(events_df.head(10))

Events: 62


,year,event_id,event_name,event_date
0,2024,6,Sony Open in Hawaii,2024-01-14
1,2024,2,The American Express,2024-01-21
2,2024,4,Farmers Insurance Open,2024-01-27
3,2024,5,AT&T Pebble Beach Pro-Am,2024-02-04
4,2024,3,WM Phoenix Open,2024-02-11
5,2024,7,The Genesis Invitational,2024-02-18
6,2024,540,Mexico Open at Vidanta,2024-02-25
7,2024,10,Cognizant Classic in The Palm Beaches,2024-03-03
8,2024,9,Arnold Palmer Invitational presented by Master...,2024-03-10
9,2024,11,THE PLAYERS Championship,2024-03-17


In [28]:
def _safe_int(x):
    try:
        return int(x)
    except Exception:
        return None

rows = []
failed = []

for i, r in events_df.iterrows():
    yr = _safe_int(r["year"])
    eid = _safe_int(r["event_id"])
    if yr is None or eid is None:
        continue

    try:
        wk = build_weekly_view(yr, eid)
        summ = wk["summary"].copy()

        # attach event metadata
        summ["year"] = yr
        summ["event_id"] = eid
        if "event_name" in r:
            summ["event_name"] = r["event_name"]
        if "event_date" in r:
            summ["event_date"] = r["event_date"]

        # field size (per event)
        summ["field_n"] = len(summ)

        rows.append(summ)

    except Exception as e:
        failed.append((yr, eid, str(e)))

master = pd.concat(rows, ignore_index=True)

print("Master rows:", len(master))
print("Failed events:", len(failed))
if failed:
    display(pd.DataFrame(failed, columns=["year","event_id","error"]).head(20))

Master rows: 7693
Failed events: 0


In [31]:
import numpy as np
import pandas as pd

# --- required columns ---
need = ["year", "event_id", "dg_id", "finish_num"]
missing = [c for c in need if c not in master.columns]
if missing:
    raise ValueError(f"master missing required columns: {missing}")

# normalize types
master["year"] = pd.to_numeric(master["year"], errors="coerce").astype("Int64")
master["event_id"] = pd.to_numeric(master["event_id"], errors="coerce").astype("Int64")
master["dg_id"] = pd.to_numeric(master["dg_id"], errors="coerce").astype("Int64")
master["finish_num"] = pd.to_numeric(master["finish_num"], errors="coerce")

# event key
master["event_key"] = master["year"].astype(str) + "_" + master["event_id"].astype(str)

# field size PER EVENT (computed from the dataset, not assumed)
master["field_n"] = master.groupby("event_key")["dg_id"].transform("nunique")

# top 10% cutoff per event
cutoff = master.groupby("event_key")["field_n"].transform(
    lambda s: int(np.ceil(0.10 * float(s.iloc[0])))
)

# label
master["is_top10pct"] = (
    (master["finish_num"].notna()) &
    (master["finish_num"] <= cutoff)
).astype(int)

print("Overall top10pct rate:", master["is_top10pct"].mean())
print("Events:", master["event_key"].nunique())
print("Rows:", len(master))

ValueError: master missing required columns: ['finish_num']